Selecting the hurdle regressor in this notebook

Imports and config

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import *

from darts import TimeSeries
from darts.dataprocessing.transformers import WindowTransformer, StaticCovariatesTransformer


warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)


In [ ]:
# -------------------- CONFIG --------------------
DATA_FOLDER       = "./data"
FIXED_DATA_PATH   = construct_path(DATA_FOLDER, "fixed")
DATASET_PATH      = construct_path(DATA_FOLDER, "dataset")

TARGET            = "act_drone_strike_on_ua"          # count target

# Forecasting setup
OUTPUT_CHUNK_LEN  = 7      # how many days ahead each model predicts in one shot
INPUT_LAGS        = 7      # how many past days of the target the model sees
MULTI_MODELS      = True   # direct strategy: one sub-model per horizon step

# CV / split
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.10, 0.20   # enforced by split_series_list
CV_STRIDE                       = 1                   # predict every day; retrain every OUTPUT_CHUNK_LEN days

# Which classifiers to benchmark in this run
REGRESSORS_TO_RUN = ["naive_last", "naive_mean",
                      "lightgbm", "lightgbm_tweedie",
                      "xgboost", "xgboost_tweedie",
                      "catboost", "catboost_tweedie",
                      'linearregression']

RANDOM_STATE      = 42
np.random.seed(RANDOM_STATE)


## 1. Load data

In [ ]:
regions,master_timeseries,regions_activity = load_data(data_path=FIXED_DATA_PATH,dataset_path=DATASET_PATH)

In [ ]:
for_global_reset, global_weather_columns = get_engineered_features(
    master_timeseries=master_timeseries,
    data_path=FIXED_DATA_PATH,
    target_col=TARGET,
    regions=regions,
    regions_activity=regions_activity,
    binarize_target=False
)

In [ ]:
print(for_global_reset.head())

In [ ]:
print(for_global_reset.isna().any()[lambda x: x])
# print(for_global_reset[for_global_reset["Activity_Level"].isna() == True][['Activity_Level','event_date','region']])

In [ ]:
# Future vs past covariate split
holiday_cols, future_covariates, exclude_cols, past_covariates = split_future_and_past_cov(for_global_reset,global_weather_columns,TARGET)

## Build Darts TimeSeries and apply windowed transforms
Getting lag
 variabels

In [ ]:
target_series_list, past_covs_list,future_covs_list = build_ts_and_apply_window_transformer(for_global_reset,TARGET,past_covariates,future_covariates,ed_alpha=halflife_to_alpha(7))

In [ ]:
full_weights = make_positive_only_weights(target_series_list)
print(full_weights)

## Encode static covariates and split 70/10/20


In [ ]:
region_names, train_target, val_target, test_target, full_past_covs, full_fut_covs, target_for_cv, TRAIN_VAL_END,CV_START_VAL =\
      get_covs_and_encodings(target_series_list,past_covs_list,future_covs_list,TRAIN_FRAC,VAL_FRAC)

## Regressors

Each branch returns a Darts forecasting classifier. The `lags_*`,
`output_chunk_length`, and encoder setup are identical across models so the
comparison is fair; only the underlying estimator hyperparameters differ.

In [ ]:


# Shared kwargs that define the forecasting structure (NOT the estimator)
COMMON_KWARGS = get_common_kwargs()


def build_regressor(name: str):
    """Return a Darts forecasting classifier ready to .fit().

    name in {'lightgbm', 'xgboost', 'catboost', 'naive'}.
    """
    name = name.lower()

    if name == "lightgbm":
        from darts.models import LightGBMModel
        return LightGBMModel(
            **COMMON_KWARGS,
            objective         = "poisson",

            num_leaves        = 31,
            max_depth         = 5,
            min_child_samples = 30,
            subsample         = 0.8,
            colsample_bytree  = 0.8,
            learning_rate     = 0.05,
            n_estimators      = 500,
            random_state      = RANDOM_STATE,
            verbose           = -1,
        )

    if name == "xgboost":
        from darts.models import XGBModel
        return XGBModel(
            **COMMON_KWARGS,
            objective         = "count:poisson",
            max_depth         = 5,
            min_child_weight  = 3,
            subsample         = 0.8,
            colsample_bytree  = 0.8,
            learning_rate     = 0.05,
            n_estimators      = 500,
            tree_method       = "hist",
            random_state      = RANDOM_STATE,
            verbosity         = 0,
        )

    if name == "catboost":
        from darts.models import CatBoostModel
        return CatBoostModel(
            **COMMON_KWARGS,
            loss_function     = "Poisson",
            boost_from_average=False ,
            depth             = 5,
            learning_rate     = 0.05,
            iterations        = 500,
            l2_leaf_reg       = 3,
            random_seed       = RANDOM_STATE,
            verbose           = False,
        )


    if name == "lightgbm_tweedie":
        from darts.models import LightGBMModel
        return LightGBMModel(
            **COMMON_KWARGS,
            objective              = "tweedie",
            tweedie_variance_power = 1.5,
            num_leaves             = 31,
            max_depth              = 5,
            min_child_samples      = 30,
            subsample              = 0.8,
            colsample_bytree       = 0.8,
            learning_rate          = 0.05,
            n_estimators           = 500,
            random_state           = RANDOM_STATE,
            verbose                = -1,
        )

    if name == "xgboost_tweedie":
        from darts.models import XGBModel
        return XGBModel(
            **COMMON_KWARGS,
            objective              = "reg:tweedie",
            tweedie_variance_power = 1.5,
            max_depth              = 5,
            min_child_weight       = 3,
            subsample              = 0.8,
            colsample_bytree       = 0.8,
            learning_rate          = 0.05,
            n_estimators           = 500,
            tree_method            = "hist",
            random_state           = RANDOM_STATE,
            verbosity              = 0,
        )

    if name == "catboost_tweedie":
        from darts.models import CatBoostModel
        return CatBoostModel(
            **COMMON_KWARGS,
            loss_function     = "Tweedie:variance_power=1.5",
            boost_from_average= False,
            depth             = 5,
            learning_rate     = 0.05,
            iterations        = 500,
            l2_leaf_reg       = 3,
            random_seed       = RANDOM_STATE,
            verbose           = False,
        )
    if name == 'linearregression':
        from darts.models import LinearRegressionModel
        return LinearRegressionModel(
            **COMMON_KWARGS,
            likelihood='poisson',
            random_state       = RANDOM_STATE,
        )

    if name == "naive":
        # Handled separately (no covariates, persistence only)
        return None

    raise ValueError(f"Unknown classifier name: {name!r}")


## Feature Selection

Fit LightGBM with binary sample weights (weight=0 for zeros, 1 for positives).

In [ ]:
model_feature_selection = build_regressor('lightgbm')
model_feature_selection.fit(
    series            = train_target,
    past_covariates   = full_past_covs,
    future_covariates = full_fut_covs,
    sample_weight     = [w.slice_intersect(ts) for w, ts in zip(full_weights, train_target)],
)

In [ ]:
gain_imps = {"Feature": model_feature_selection.lagged_feature_names}

underlying_model = model_feature_selection.model
estimators = underlying_model.estimators_ if hasattr(underlying_model, "estimators_") else [underlying_model]
for h, est in enumerate(estimators, start=1):
    gain_imps[f"h{h}_gain"] = est.booster_.feature_importance(importance_type="gain")

df_gain = pd.DataFrame(gain_imps)
df_gain["mean_gain"] = df_gain.filter(like="_gain").mean(axis=1)
df_gain = df_gain.sort_values("mean_gain", ascending=False).reset_index(drop=True)
df_gain.to_csv("prelimFeatureImportanceRegressor.csv", index=False)
print(df_gain.head(20))

In [ ]:
top_100_features      = df_gain.head(100)["Feature"].to_list()
top_100_features_dict = clean_feature_names(top_100_features)

In [ ]:
past_covs_list   = [subset_safe(ts, top_100_features_dict['pastcov_features_base']) for ts in past_covs_list]
future_covs_list = [subset_safe(ts, top_100_features_dict['futcov_features_base'])  for ts in future_covs_list]

region_names, train_target, val_target, test_target, full_past_covs, full_fut_covs, target_for_cv,TRAIN_VAL_END,CV_START_VAL = \
    get_covs_and_encodings(target_series_list,past_covs_list,future_covs_list,TRAIN_FRAC,VAL_FRAC)

In [ ]:
sample = past_covs_list[0]
base_set = top_100_features_dict['pastcov_features_base']
present  = base_set & set(sample.components)
missing  = base_set - set(sample.components)
print(f"{len(present)}/{len(base_set)} base features present in past_covs")
if missing:
    print("missing (first 10):", list(missing)[:10])
sample = future_covs_list[0]
base_set = top_100_features_dict['futcov_features_base']
present  = base_set & set(sample.components)
missing  = base_set - set(sample.components)
print(f"{len(present)}/{len(base_set)} base features present in future_covs")
if missing:
    print("missing (first 10):", list(missing)[:10])

##  Evaluation utilities

Two views of quality, all from one pass over the predictions:

| scope              | granularity                           |
| ------------------ | ------------------------------------- |
| **per-region**     | one row per region                     |
| **global**         | pooled across all regions AND horizons |

Regressor metrics:
- `RMSE` and `mae`


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import mean_poisson_deviance

def _regressor_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float).ravel()
    y_pred = np.clip(np.asarray(y_pred).astype(float).ravel(), 0, None)  # counts are ≥ 0
    pos = y_true > 0  # evaluate the *conditional* fit on positives only
    return {
        "RMSE_all":      np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE_all":       mean_absolute_error(y_true, y_pred),
        "RMSE_pos":      np.sqrt(mean_squared_error(y_true[pos], y_pred[pos])) if pos.any() else np.nan,
        "MAE_pos":       mean_absolute_error(y_true[pos], y_pred[pos]) if pos.any() else np.nan,
        "PoissonDev_pos":mean_poisson_deviance(y_true[pos], np.clip(y_pred[pos], 1e-6, None)) if pos.any() else np.nan,
        "n":             int(len(y_true)),
        "n_positive":    int(pos.sum()),
    }

# collect_predictions_long imported from src

def evaluate_long(long_df):
    def _rows_from_groupby(group_cols):
        rows = []
        for keys, sub in long_df.groupby(group_cols):
            m = _regressor_metrics(sub["y_true"], sub["y_pred"])
            if isinstance(group_cols, str):
                m[group_cols] = keys
            else:
                for k, v in zip(group_cols, keys if isinstance(keys, tuple) else (keys,)):
                    m[k] = v
            rows.append(m)
        return pd.DataFrame(rows)

    # Fixed capitalization here:
    per_region         = _rows_from_groupby("region").sort_values("RMSE_pos", ascending=False).reset_index(drop=True)
    per_horizon        = _rows_from_groupby("horizon").sort_values("horizon").reset_index(drop=True)
    per_region_horizon = _rows_from_groupby(["region", "horizon"]).sort_values(["region", "horizon"]).reset_index(drop=True)
    global_row         = _regressor_metrics(long_df["y_true"], long_df["y_pred"])

    return {
        "per_region":         per_region,
        "per_horizon":        per_horizon,
        "per_region_horizon": per_region_horizon,
        "global":             global_row,
    }

In [ ]:
# plot_region_horizon_heatmap imported from src


## Naive persistence baseline

This predicts `y_hat(t+h) = y(t)` for all h. (Hewamalage et al., 2023).
We emit it in the same long-format so it plugs into the same evaluator. Here we try last non negative or non negative mean


In [ ]:
# naive_historical_forecasts imported from src

def naive_collect_long(target_list, region_names, start_frac, horizon=OUTPUT_CHUNK_LEN,
                       stride=CV_STRIDE, method="last_non_negative"):
    """Collects predictions into a long DataFrame for the specified baseline method."""
    fold_preds_list = [
        naive_historical_forecasts(ts, start_frac, horizon, stride, method=method)
        for ts in target_list
    ]
    return collect_predictions_long(target_list, fold_preds_list, region_names), fold_preds_list

In [ ]:
def run_manual_expanding_cv(name, past_covs_override=None, fut_covs_override=None):
    """Manual expanding-window CV with decoupled predict / retrain strides.

    Predicts every CV_STRIDE days; retrains every OUTPUT_CHUNK_LEN days.
    Between retrains the frozen model predicts from a growing context window.

    Returns
    -------
    list[list[TimeSeries]]
        Outer list indexed by region, inner list by prediction step.
    """
    ref_ts    = target_for_cv[0]            # all regions share the same time index
    n_total   = len(ref_ts)
    start_idx = int(CV_START_VAL * n_total)

    n_regions      = len(target_for_cv)
    all_fold_preds = [[] for _ in range(n_regions)]   # outer=region, inner=prediction step
    n_preds    = 0
    n_retrains = 0
    model      = None

    for t0 in range(start_idx, n_total - OUTPUT_CHUNK_LEN + 1, CV_STRIDE):
        steps_since_start = t0 - start_idx
        split_time        = ref_ts.time_index[t0]

        _pc = past_covs_override if past_covs_override is not None else full_past_covs
        _fc = fut_covs_override  if fut_covs_override  is not None else full_fut_covs

        if steps_since_start % OUTPUT_CHUNK_LEN == 0:
            train_series  = [ts.drop_after(split_time) for ts in target_for_cv]
            train_weights = [w.drop_after(split_time)  for w  in full_weights]
            model = build_regressor(name)
            model.fit(
                series            = train_series,
                past_covariates   = _pc,
                future_covariates = _fc,
                sample_weight     = train_weights,
            )
            n_retrains += 1
            print(f"   retrain {n_retrains}  (data up to {split_time.date()})")

        pred_series = [ts.drop_after(split_time) for ts in target_for_cv]

        preds = model.predict(
            n                 = OUTPUT_CHUNK_LEN,
            series            = pred_series,
            past_covariates   = _pc,
            future_covariates = _fc,
            show_warnings     = False,
        )

        # preds is list[TimeSeries], one per region
        for r_idx, pred in enumerate(preds):
            all_fold_preds[r_idx].append(pred)

        n_preds += 1

    print(f"   {n_preds} predictions, {n_retrains} retrains complete")
    return all_fold_preds

In [21]:
# One pass: fit + CV + long-format predictions per model
long_by_model       = {}   # name -> long DataFrame of (region, fold, horizon, y_true, y_prob)
fold_preds_by_model = {}   # name -> list[list[TimeSeries]] (kept for plotting)

for name in REGRESSORS_TO_RUN:
    print(f"\n=== Training + CV: {name} ===")
    
    if name == "naive_last":
        long_df, fold_preds = naive_collect_long(
            target_for_cv, region_names, start_frac=CV_START_VAL, method="last_non_negative"
        )
    elif name == "naive_mean":
        long_df, fold_preds = naive_collect_long(
            target_for_cv, region_names, start_frac=CV_START_VAL, method="non_negative_mean"
        )
    else:
        fold_preds = run_manual_expanding_cv(name)
        long_df    = collect_predictions_long(target_for_cv, fold_preds, region_names)

    long_by_model[name]       = long_df
    fold_preds_by_model[name] = fold_preds
    print(f"   collected {len(long_df):,} (region,fold,horizon) prediction points")

   fold 4 done  (trained up to 2024-06-01)
   fold 8 done  (trained up to 2024-06-29)
   fold 12 done  (trained up to 2024-07-27)
   12 folds complete
   collected 1,680 (region,fold,horizon) prediction points

=== Training + CV: catboost_tweedie ===
   fold 1 done  (trained up to 2024-05-11)
   fold 4 done  (trained up to 2024-06-01)
   fold 8 done  (trained up to 2024-06-29)
   fold 12 done  (trained up to 2024-07-27)
   12 folds complete
   collected 1,680 (region,fold,horizon) prediction points

=== Training + CV: linearregression ===


/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_glm/glm.py:285: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_glm/glm.py:285: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or s

   fold 1 done  (trained up to 2024-05-11)


/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_glm/glm.py:285: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_glm/glm.py:285: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or s

   fold 4 done  (trained up to 2024-06-01)


/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_glm/glm.py:285: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_glm/glm.py:285: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or s

   fold 8 done  (trained up to 2024-06-29)


/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_glm/glm.py:285: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_glm/glm.py:285: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or s

   fold 12 done  (trained up to 2024-07-27)
   12 folds complete
   collected 1,680 (region,fold,horizon) prediction points


/home/jan/thesis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_glm/glm.py:285: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)


## 10. Leaderboard - which regressor wins overall?

In [22]:
results_by_model = {}
for name, long_df in long_by_model.items():
    res = evaluate_long(long_df)
    results_by_model[name] = res

In [23]:
leaderboard_rows = []
for name, res in results_by_model.items():
    g = res["global"]
    leaderboard_rows.append({
        "model":     name,
        "RMSE_all":        g["RMSE_all"],
        "MAE_all": g["MAE_all"],
        "RMSE_pos":    g["RMSE_pos"],
        "MAE_pos":   g["MAE_pos"],
        "PoissonDev_pos":    g["PoissonDev_pos"],
    })
leaderboard = pd.DataFrame(leaderboard_rows).sort_values("RMSE_pos", ascending=True).reset_index(drop=True)
leaderboard


,model,RMSE_all,MAE_all,RMSE_pos,MAE_pos,PoissonDev_pos
0,catboost_tweedie,1.517196,1.301110,1.968695,1.293524,1.094980
1,catboost,1.545434,1.330518,1.992623,1.314472,1.107875
2,xgboost,1.544072,1.328267,2.005023,1.312124,1.123261
3,xgboost_tweedie,1.513165,1.294260,2.049110,1.339067,1.203642
4,lightgbm,1.585298,1.340505,2.088818,1.355621,1.244479
5,lightgbm_tweedie,1.530975,1.291499,2.107867,1.350687,1.307701
6,linearregression,2.097785,1.938009,2.408499,1.628062,1.810564
7,naive_last,1.454468,0.595238,2.806378,1.936364,20.792996
8,naive_mean,1.362982,0.521726,3.035470,2.202718,5.703209


## 13. Per-region detail for the winning model

In [24]:
results_by_model['catboost_tweedie']["per_region"]


,RMSE_all,MAE_all,RMSE_pos,MAE_pos,PoissonDev_pos,n,n_positive,region
0,3.331822,2.558133,3.325706,2.506780,2.490456,84,79,sumy
1,2.837969,2.489034,1.929275,1.592318,1.055551,84,46,zaporizhia
2,1.557217,1.355305,1.488523,1.183961,0.992987,84,53,dnipropetrovsk
3,1.207048,1.064566,1.164555,0.868000,0.725456,84,37,chernihiv
4,1.287729,1.149953,1.023415,0.809770,0.541009,84,43,kherson
5,1.219591,1.108961,0.800525,0.568242,0.376923,84,27,kharkiv
6,1.185844,1.175949,0.667820,0.650973,0.316845,84,2,lviv
7,1.172170,1.153410,0.385558,0.326756,0.110477,84,3,odesa
8,1.154936,1.117387,0.367389,0.275599,0.101071,84,7,donetsk
9,1.467472,1.430575,0.341566,0.329557,0.094106,84,3,zhytomyr


In [25]:
results_by_model['catboost_tweedie']["per_horizon"]

,RMSE_all,MAE_all,RMSE_pos,MAE_pos,PoissonDev_pos,n,n_positive,horizon
0,1.369266,1.215578,1.726073,1.113793,0.859095,240,49,1
1,1.515753,1.335982,1.824454,1.280143,1.045103,240,33,2
2,1.474595,1.319775,1.770378,1.323517,1.115193,240,40,3
3,1.602176,1.337645,2.198704,1.396134,1.260125,240,44,4
4,1.548108,1.299724,2.058836,1.330748,1.074223,240,52,5
5,1.577184,1.317850,2.206704,1.381639,1.252661,240,48,6
6,1.521596,1.281217,1.899566,1.252410,1.073733,240,64,7


## 14. Feature importance for the winning model (tree models only)

For multi-output direct classifiers, each horizon has its own sub-estimator,
so we emit importance per horizon day. The first column (`h=1`) is usually
the most trustworthy - longer horizons spread importance over more lagged
features.


In [26]:
# feature_importances_per_horizon imported from src
